# EdgeTRM (Official TRM path, notebook-friendly)

This notebook uses the **official TinyRecursiveModels TRM + ACT loss** through a thin adapter, instead of re-implementing recursion in-notebook.

Goal: provide a small, runnable baseline where loss decreases reliably, then you can swap in Sudoku/ARC data.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
trm_root = repo_root / "TinyRecursiveModels"
if str(trm_root) not in sys.path:
    sys.path.insert(0, str(trm_root))

print("repo_root:", repo_root)
print("trm_root:", trm_root)

In [ ]:
import torch
from edge_trm_wrapper import EdgeTRMAdapter, EdgeTRMBatch

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1) Tiny config (fast sanity training)

This is intentionally tiny and uses `halt_max_steps=1` so each iteration is straightforward for debugging.

In [ ]:
cfg = {
    "batch_size": 32,
    "seq_len": 12,
    "puzzle_emb_ndim": 0,
    "num_puzzle_identifiers": 1,
    "vocab_size": 16,
    "H_cycles": 1,
    "L_cycles": 1,
    "H_layers": 0,
    "L_layers": 1,
    "hidden_size": 64,
    "expansion": 2.0,
    "num_heads": 4,
    "pos_encodings": "rope",
    "halt_max_steps": 1,
    "halt_exploration_prob": 0.0,
    "forward_dtype": "float32",
    "mlp_t": False,
    "puzzle_emb_len": 0,
    "no_ACT_continue": True,
}

model = EdgeTRMAdapter(cfg, loss_type="stablemax_cross_entropy").to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
print("params:", sum(p.numel() for p in model.parameters()))

## 2) Synthetic task

Identity token reconstruction (`labels = inputs`).

If the training path is wired correctly, the loss should trend down quickly.

In [ ]:
def make_batch(batch_size=32, seq_len=12, vocab_size=16, device=device):
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    y = x.clone()
    puzzle_ids = torch.zeros((batch_size,), dtype=torch.int32, device=device)
    return EdgeTRMBatch(inputs=x, labels=y, puzzle_identifiers=puzzle_ids)

In [ ]:
model.train()
losses = []
lm_losses = []
accs = []

# fixed batch so you can verify optimization wiring by overfitting
batch = make_batch(cfg["batch_size"], cfg["seq_len"], cfg["vocab_size"])
carry = model.initial_carry(batch)
# Move carry tensors to selected device
carry.inner_carry.z_H = carry.inner_carry.z_H.to(device)
carry.inner_carry.z_L = carry.inner_carry.z_L.to(device)
carry.steps = carry.steps.to(device)
carry.halted = carry.halted.to(device)
carry.current_data = {k: v.to(device) for k, v in carry.current_data.items()}

for step in range(300):
    carry, loss, metrics, _, done = model.train_step(batch=batch, carry=carry)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    losses.append(float(loss.detach().cpu()))
    lm_losses.append(float((metrics["lm_loss"] / metrics["count"]).detach().cpu()))
    accs.append(float((metrics["accuracy"] / metrics["count"]).detach().cpu()))

    if step % 50 == 0:
        print(
            f"step={step:03d} total={losses[-1]:.4f} lm={lm_losses[-1]:.4f} tok_acc={accs[-1]:.4f}"
        )

print("start_lm_loss:", lm_losses[0], "end_lm_loss:", lm_losses[-1])
print("start_acc:", accs[0], "end_acc:", accs[-1])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(lm_losses)
ax[0].set_title("LM loss per token")
ax[0].set_xlabel("step")
ax[0].set_ylabel("loss")
ax[0].grid(alpha=0.2)

ax[1].plot(accs)
ax[1].set_title("Token accuracy")
ax[1].set_xlabel("step")
ax[1].set_ylabel("accuracy")
ax[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 3) Next step for your real use-case

Replace `make_batch(...)` with your Sudoku/ARC batch function, while keeping:

- `EdgeTRMBatch(inputs, labels, puzzle_identifiers)`
- `carry = model.initial_carry(...)` once
- recurrent `carry, loss, metrics, ... = model.train_step(...)`

That keeps recursion + ACT loss behavior aligned with the official implementation.